# Ingesta Bronze · Xbox

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql.window import Window
from datetime import datetime, timezone
import csv
import re
DEFAULTS = {'storageName': 'adlsmardata1412jmvcc', 'catalogo': 'fi_videojuegos', 'storageCredential': 'credential', 'container_metastore': 'metastore', 'carpeta_proyecto': 'fi_videojuegos_jmcc', 'raw_subdir': '', 'aplicar_grants': 'false'}
for clave, valor in DEFAULTS.items():
    dbutils.widgets.text(clave, valor)
PARAMS = {clave: dbutils.widgets.get(clave) for clave in DEFAULTS}
CATALOGO = PARAMS['catalogo'].strip()
STORAGE = PARAMS['storageName'].strip()
CARPETA_PROYECTO = PARAMS['carpeta_proyecto'].strip('/')
raw_subdir = PARAMS['raw_subdir'].strip('/')
assert re.fullmatch('[a-z][a-z0-9_]*', CATALOGO)
assert re.fullmatch('[a-z0-9]{3,24}', STORAGE)
assert re.fullmatch('[a-z0-9_/-]+', CARPETA_PROYECTO) and '..' not in CARPETA_PROYECTO
assert not raw_subdir or (re.fullmatch('[A-Za-z0-9_/-]+', raw_subdir) and '..' not in raw_subdir)
RUTA_RAW = f'abfss://raw@{STORAGE}.dfs.core.windows.net/' + raw_subdir
RUTA_RAW = RUTA_RAW.rstrip('/')
TABLAS_EXTERNAS = True

def q(s):
    return '`' + s.replace('`', '``') + '`'

def nombre(capa, tabla):
    assert capa in ('bronze', 'silver', 'golden')
    assert tabla.startswith('fi_'), 'Solo se permiten tablas de este proyecto.'
    return f'{q(CATALOGO)}.{q(capa)}.{q(tabla)}'

def ruta_delta(capa, tabla):
    return f'abfss://{capa}@{STORAGE}.dfs.core.windows.net/{CARPETA_PROYECTO}/{tabla}'

def preparar(df, capa, tabla):
    destino = nombre(capa, tabla)
    actual = spark.table(destino)
    assert [(f.name, f.dataType) for f in actual.schema] == [(f.name, f.dataType) for f in df.schema], f'Esquema diferente en {destino}. Revisar antes de reemplazar datos.'
    detalle = spark.sql(f'DESCRIBE DETAIL {destino}').first().asDict()
    assert detalle['format'].lower() == 'delta'
    if TABLAS_EXTERNAS:
        assert detalle['location'].rstrip('/') == ruta_delta(capa, tabla).rstrip('/'), f'Tabla ya registrada en otra ruta: {destino}. No se modifica.'
    else:
        tipo = spark.sql(f'DESCRIBE TABLE EXTENDED {destino}').collect()
        tipos = [r['data_type'] for r in tipo if r['col_name'].strip() == 'Type']
        assert tipos and tipos[0].upper() == 'MANAGED', f'{destino} ya es externa.'
    return destino

def guardar(df, capa, tabla):
    destino = preparar(df, capa, tabla)
    df.select(*spark.table(destino).columns).write.mode('overwrite').insertInto(destino)
    resultado = spark.table(destino)
    assert resultado.count() == df.count(), f'Conteo diferente: {destino}'
    print('OK', destino, resultado.count(), 'filas')
    return resultado

In [0]:
FUENTES = [{'file': 'nintendo.csv', 'table': 'fi_br_nintendo', 'columns': ['game_title', 'normalized_title', 'platform', 'platform_original', 'platform_vgsales_code', 'platform_cleaning_note', 'release_date_original', 'release_date', 'release_year', 'release_month', 'release_month_name_es', 'release_day', 'release_quarter', 'date_status', 'meta_score', 'user_score', 'esrb_rating', 'link', 'developer_count', 'developer_1', 'developer_2', 'developer_3', 'developer_4', 'developers_original', 'genre_count', 'genre_1', 'genre_2', 'genre_3', 'genre_4', 'genre_5', 'genre_6', 'genre_7', 'genre_8', 'genre_9', 'genre_10', 'genres_original', 'join_key_vgsales'], 'rows': 1026, 'sha256': '9130cb5af6d344b0af4553739da7a16328e528f4e0f2342e1c6f344e5845f705'}, {'file': 'playstation.csv', 'table': 'fi_br_playstation', 'columns': ['id_igdb', 'nombre_videojuego', 'fecha_primer_lanzamiento', 'anio_primer_lanzamiento', 'id_plataforma_igdb', 'plataforma', 'codigo_plataforma_vgsales', 'coincide_codigo_vgsales', 'titulo_normalizado', 'clave_join_nombre_plataforma', 'clave_join_nombre_anio_plataforma', 'todas_las_plataformas', 'ids_todas_las_plataformas', 'generos', 'ids_generos', 'calificacion_igdb_0_100', 'puntuacion_hltb', 'cantidad_resenas_hltb', 'personas_encuestadas_hltb', 'horas_historia_principal', 'horas_historia_mas_extras', 'horas_completista_100', 'resumen_original', 'historia_original'], 'rows': 31002, 'sha256': '08292466a360602b1ed478fa59c48e79b4b6d132f3d598f543ef602eb4604c9e'}, {'file': 'vgsales.csv', 'table': 'fi_br_vgsales', 'columns': ['Rank', 'Name', 'Platform', 'Year', 'Genre', 'Publisher', 'NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales', 'Global_Sales'], 'rows': 16598, 'sha256': 'e2076095ffcae2a92dbc6de6ecbd54455ee034bb38da550b2dc385e9265d4ebe'}, {'file': 'xbox.csv', 'table': 'fi_br_xbox', 'columns': ['id_igdb', 'nombre_videojuego', 'fecha_primer_lanzamiento', 'anio_primer_lanzamiento', 'id_plataforma_igdb', 'plataforma', 'codigo_plataforma_vgsales', 'coincide_codigo_vgsales', 'titulo_normalizado', 'clave_join_nombre_plataforma', 'clave_join_nombre_anio_plataforma', 'todas_las_plataformas', 'ids_todas_las_plataformas', 'generos', 'ids_generos', 'calificacion_igdb_0_100', 'puntuacion_hltb', 'cantidad_resenas_hltb', 'personas_encuestadas_hltb', 'horas_historia_principal', 'horas_historia_mas_extras', 'horas_completista_100', 'resumen_original', 'historia_original'], 'rows': 17993, 'sha256': '85300b4b4765c0f502fb20e958079ac2365cc39461d23b200066f1411aef2219'}]

In [0]:
OPCIONES = dict(header="true", sep=",", encoding="UTF-8", quote='"', escape='"',
    multiLine="true", mode="FAILFAST", enforceSchema="false",
    ignoreLeadingWhiteSpace="false", ignoreTrailingWhiteSpace="false")
validados = {}
resumen_bronze = {}
fecha_carga = datetime.now(timezone.utc)

def validar_bronze(indice):
    fuente = FUENTES[indice]
    ruta = f"{RUTA_RAW}/{fuente['file']}"
    # ABFSS se lee con Spark, no con open() ni con un volumen.
    binario = spark.read.format('binaryFile').load(ruta).select(F.sha2('content',256).alias('sha256'))
    huellas = binario.collect()
    assert len(huellas) == 1 and huellas[0]['sha256'] == fuente['sha256'], f"Archivo cambiado: {ruta}"
    esquema = StructType([StructField(c, StringType(), True) for c in fuente['columns']])
    df = spark.read.options(**OPCIONES).schema(esquema).csv(ruta)
    # Evaluar todos los campos, no solamente count(), para activar el parser CSV.
    est = df.agg(F.count(F.lit(1)).alias('filas'),
        *[F.count(F.col(q(c))).alias(f'c{i}') for i,c in enumerate(fuente['columns'])]).first()
    assert est['filas'] == fuente['rows'], f"Conteo inesperado: {ruta}"
    validados[indice] = df
    print('Validado:', ruta, est['filas'], 'filas')


In [0]:
def cargar_bronze(indice):
    assert indice in validados, "Validar esta fuente antes de cargarla."
    fuente = FUENTES[indice]
    df = (validados[indice].withColumn("fi_fecha_ingesta", F.lit(fecha_carga))
          .withColumn("fi_archivo_origen", F.lit(fuente["file"])))
    destino = guardar(df, "bronze", fuente["table"])
    assert destino.count() == fuente["rows"]
    resumen_bronze[indice] = (fuente["file"], fuente["table"], destino.count(), "OK")
    destino.printSchema()
    display(destino.limit(5))

In [0]:
validar_bronze(3)
cargar_bronze(3)